# Nạp chunk vào IVFPQ index đã train — Kaggle / Colab

Input: `index.faiss` từ `FAISS_IVFPQ_Index.ipynb` (đã train, chưa thêm vector) và `merged.json`: mảng tài liệu có `doc_id`, `status`, `url`, `parents`, `children`. Mỗi child chứa một chunk và các ID liên quan. Notebook đọc JSON theo luồng, kể cả JSON xuống dòng hoặc viết gọn; không tải cả dataset vào RAM.

Output: `index.faiss` và `metadata.json` trong `output_dir`. Giữ nguyên index/dataset đầu vào. File đang ghi dùng đuôi `.partial`; chỉ đổi sang tên cuối sau khi toàn bộ quá trình thành công. Nếu bị ngắt, giữ file `.partial` để kiểm tra và chọn thư mục output mới khi chạy lại; notebook chưa hỗ trợ resume.

**Trước khi chạy:**
- Kaggle: bật GPU và Internet; gắn dataset rồi sửa `index_path`, `data_path` ở cell cấu hình.
- Colab: chọn runtime GPU; upload/mount dữ liệu rồi sửa hai đường dẫn đó. Output mặc định là `/content/faiss_chunks`, cần tải xuống hoặc lưu vào Drive trước khi runtime kết thúc.
- Nếu runtime đã báo `device-side assert triggered`, **Restart session/runtime rồi chạy từ cell đầu**. Chạy lại cell hoặc `empty_cache()` không khôi phục CUDA context đã lỗi.
- Lần đầu tải model cần Internet và khoảng 630 MB dữ liệu model/tokenizer. Notebook tăng timeout tải HTTP; nếu đã import Hugging Face trước đó, restart để áp dụng timeout mới.
- Chạy các cell từ trên xuống. Không tự cài/đổi phiên bản PyTorch, CUDA hoặc driver. Cell tiếp theo chỉ kiểm tra dependency. Khi thiếu gói, tự chạy đúng lệnh `%pip install ...` được báo sau khi bạn quyết định cài, rồi restart runtime nếu đã import thư viện.

Dữ liệu lớn (khoảng 19 triệu chunk) cần đủ thời gian, RAM cho index và dung lượng output. Dùng `start_point`/`end_point` để chạy thử một khoảng nhỏ trước; notebook không đảm bảo hoàn thành toàn bộ dữ liệu trong giới hạn phiên miễn phí.

## 1. Kiểm tra môi trường và cấu hình

In [ ]:
import importlib.util
import sys

required_packages = {
    "numpy": "numpy",
    "torch": "torch",
    "faiss": "faiss-cpu",
    "sentence_transformers": "sentence-transformers",
}
missing_packages = [
    package for module, package in required_packages.items()
    if importlib.util.find_spec(module) is None
]
if missing_packages:
    raise ModuleNotFoundError(
        "Missing notebook dependencies: " + ", ".join(missing_packages)
        + ". No packages were installed. If approved, run: %pip install "
        + " ".join(missing_packages)
    )
print(f"Python: {sys.version.split()[0]}; dependency discovery passed")

In [ ]:
import gc
import json
import os
from importlib.metadata import version
from pathlib import Path
from types import MethodType
from typing import Iterator, TextIO

# Configure Hub timeouts before importing libraries that initialize the Hub client.
os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "120")
os.environ.setdefault("HF_HUB_ETAG_TIMEOUT", "30")

# Set this before importing torch; restart first if debugging a failed CUDA context.
debug_cuda = False
if debug_cuda:
    os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

import faiss
import numpy as np
import torch
from sentence_transformers import SentenceTransformer

if Path("/kaggle/working").is_dir():
    runtime_dir = Path("/kaggle/working")
    input_dir = Path("/kaggle/input/YOUR_DATASET")
elif Path("/content").is_dir():
    runtime_dir = Path("/content")
    input_dir = runtime_dir
else:
    runtime_dir = Path.cwd()
    input_dir = runtime_dir

index_path = input_dir / "index.faiss"  # Edit these two paths for your dataset.
data_path = input_dir / "merged.json"
output_dir = runtime_dir / "faiss_chunks"

start_point = 0
end_point = -1  # Inclusive doc_id range; -1 means no upper bound.
embedding_model = "Alibaba-NLP/gte-multilingual-base"
model_code_revision = "40ced75c3017eb27626c9d4ea981bde21a2662f4"
batch_size = 32
add_batch_size = 1_024
use_fp16 = True
use_faiss_gpu = False  # CPU FAISS works with GPU embeddings and avoids GPU build issues.
require_gpu = False  # Set True to fail early instead of using CPU embeddings.

if not isinstance(batch_size, int) or batch_size < 1:
    raise ValueError("batch_size must be a positive integer")
if not isinstance(add_batch_size, int) or add_batch_size < 1:
    raise ValueError("add_batch_size must be a positive integer")
if start_point < 0 or (end_point != -1 and end_point < start_point):
    raise ValueError("Use start_point >= 0 and end_point = -1 or >= start_point")
for input_path in (index_path, data_path):
    if not input_path.is_file():
        raise FileNotFoundError(f"Input file not found; edit the configuration: {input_path}")

output_index = output_dir / "index.faiss"
output_metadata = output_dir / "metadata.json"
partial_index = output_dir / "index.faiss.partial"
partial_metadata = output_dir / "metadata.json.partial"
for output_path in (output_index, output_metadata, partial_index, partial_metadata):
    if output_path.resolve() in (index_path.resolve(), data_path.resolve()):
        raise ValueError("Output paths must differ from input paths")
    if output_path.exists():
        raise FileExistsError(f"Choose a new output_dir to preserve existing files: {output_path}")
output_dir.mkdir(parents=True, exist_ok=True)
print(f"torch={torch.__version__}; CUDA runtime={torch.version.cuda}; faiss={faiss.__version__}")
print(f"transformers={version('transformers')}; sentence-transformers={version('sentence-transformers')}")

## 2. Đọc index và tải model

Giữ nguyên `Alibaba-NLP/gte-multilingual-base`, pooling của SentenceTransformer và chuẩn hóa L2 như khi train. Không tự giảm giới hạn 8192 token; chuỗi dài vẫn được tokenizer truncation theo giới hạn model. `use_fp16=True` chỉ dùng autocast trên CUDA; khi embedding không hữu hạn, thử lại bằng FP32 và báo rõ. Khi hết VRAM, giảm batch dần đến 1, không bỏ chunk hay tự chuyển sang CPU.

Mã GTE được cố định ở revision trong traceback. Tải trên CPU, tái tạo `position_ids` và RoPE bằng cấu hình gốc trước khi chuyển GPU: các buffer không lưu trong checkpoint có thể chưa khởi tạo đúng với Transformers 5. Bổ sung helper attention mask chỉ khi Transformers không còn cung cấp helper cũ. Token ID, position ID và shape được kiểm tra trên CPU trước mỗi lượt forward. Chạy smoke test cả CPU và thiết bị thực tế trước khi tạo metadata.

FAISS mặc định chạy CPU để dùng được `faiss-cpu` trên cả Kaggle/Colab; embedding vẫn dùng GPU nếu có. Chỉ bật `use_faiss_gpu` khi đã có bản FAISS GPU tương thích. PQ GPU dùng FP16 theo [tài liệu FAISS](https://github.com/facebookresearch/faiss/wiki/Faiss-on-the-GPU), và notebook báo rõ nếu chuyển index sang GPU thất bại.

In [ ]:
def prepare_gte_model(auto_model: torch.nn.Module) -> None:
    """Restore GTE's non-persistent buffers on CPU without changing learned weights."""
    if auto_model.config.model_type != "new":
        raise ValueError("This compatibility repair is specific to Alibaba GTE NewModel")
    embeddings = auto_model.embeddings
    expected_positions = torch.arange(
        auto_model.config.max_position_embeddings, dtype=torch.long, device="cpu",
    )
    embeddings.register_buffer("position_ids", expected_positions, persistent=False)
    # Reuse GTE's own initializer, including the original NTK scaling parameters.
    with torch.device("cpu"):
        embeddings._init_rope(auto_model.config)
    for name, buffer in embeddings.rotary_emb.named_buffers():
        if buffer.device.type != "cpu" or not torch.isfinite(buffer).all():
            raise ValueError(f"Invalid RoPE buffer after initialization: {name}")
    rotary = embeddings.rotary_emb
    if rotary.cos_cached.shape[0] < len(expected_positions):
        raise ValueError("RoPE cache is shorter than the configured position range")
    if not torch.equal(embeddings.position_ids, expected_positions):
        raise ValueError("GTE position_ids initialization failed")

    if not hasattr(auto_model, "get_extended_attention_mask"):
        def get_encoder_attention_mask(
            self, attention_mask: torch.Tensor, input_shape: tuple,
            device: torch.device | None = None, dtype: torch.dtype | None = None,
        ) -> torch.Tensor:
            """Reproduce the legacy bidirectional encoder padding mask."""
            if getattr(self.config, "is_decoder", False):
                raise ValueError("This compatibility helper supports encoder models only")
            if attention_mask.ndim == 2:
                extended_mask = attention_mask[:, None, None, :]
            elif attention_mask.ndim == 3:
                extended_mask = attention_mask[:, None, :, :]
            else:
                raise ValueError(f"Unsupported attention mask shape: {attention_mask.shape}")
            mask_dtype = self.dtype if dtype is None else dtype
            extended_mask = extended_mask.to(dtype=mask_dtype)
            return (1.0 - extended_mask) * torch.finfo(mask_dtype).min

        auto_model.get_extended_attention_mask = MethodType(
            get_encoder_attention_mask, auto_model,
        )


def validate_features(features: dict, auto_model: torch.nn.Module) -> None:
    """Reject invalid embedding indices on CPU before launching CUDA kernels."""
    input_ids = features["input_ids"]
    if input_ids.device.type != "cpu" or input_ids.dtype != torch.long:
        raise ValueError("Token IDs must be CPU int64 tensors before validation")
    if input_ids.ndim != 2 or not input_ids.numel():
        raise ValueError("Expected non-empty input_ids with shape (batch, sequence)")
    sequence_length = input_ids.shape[1]
    if sequence_length > auto_model.config.max_position_embeddings:
        raise ValueError("Token sequence exceeds the model's position limit")
    vocab_size = auto_model.get_input_embeddings().num_embeddings
    if input_ids.min().item() < 0 or input_ids.max().item() >= vocab_size:
        raise ValueError(f"Tokenizer IDs are outside the embedding vocabulary [0, {vocab_size})")
    mask = features.get("attention_mask")
    if mask is None or mask.shape != input_ids.shape or mask.device.type != "cpu":
        raise ValueError("attention_mask must match input_ids on CPU")
    if not torch.all((mask == 0) | (mask == 1)) or not torch.all(mask.sum(1) > 0):
        raise ValueError("Each attention_mask row must contain valid unmasked tokens")
    # Explicit positions also protect forward from a stale internal position buffer.
    features["position_ids"] = torch.arange(sequence_length).expand_as(input_ids)
    type_vocab_size = auto_model.config.type_vocab_size
    if type_vocab_size == 1:
        features["token_type_ids"] = torch.zeros_like(input_ids)
    elif type_vocab_size > 1 and "token_type_ids" in features:
        token_types = features["token_type_ids"]
        if (token_types.shape != input_ids.shape or token_types.dtype != torch.long
                or token_types.min().item() < 0 or token_types.max().item() >= type_vocab_size):
            raise ValueError("Invalid token_type_ids")


def embed_once(texts: list[str], mixed_precision: bool) -> np.ndarray:
    """Tokenize on CPU and run the existing SentenceTransformer pooling modules."""
    if callable(getattr(model, "preprocess", None)):
        features = model.preprocess(texts)
    else:
        features = model.tokenize(texts)
    validate_features(features, auto_model)
    features = {
        name: value.to(device) if isinstance(value, torch.Tensor) else value
        for name, value in features.items()
    }
    with torch.inference_mode(), torch.autocast(
        device_type=device.type, dtype=torch.float16,
        enabled=mixed_precision and device.type == "cuda",
    ):
        # This retains all configured pooling/normalization modules from the checkpoint.
        sentence_embeddings = model(features)["sentence_embedding"].float()
    vectors = sentence_embeddings.cpu().numpy()
    vectors = np.ascontiguousarray(vectors, dtype=np.float32)
    if vectors.shape != (len(texts), index.d):
        raise ValueError(f"Unexpected embedding shape: {vectors.shape}; expected {(len(texts), index.d)}")
    if not np.isfinite(vectors).all():
        raise FloatingPointError("Model produced non-finite embeddings")
    norms = np.linalg.norm(vectors, axis=1)
    if np.any(norms <= 1e-12):
        raise FloatingPointError("Model produced zero embeddings")
    faiss.normalize_L2(vectors)
    return vectors


def encode_texts(texts: list[str]) -> np.ndarray:
    """Retry CUDA OOM with smaller batches, preserving every input and its order."""
    global effective_batch_size
    vectors = np.empty((len(texts), index.d), dtype=np.float32)
    # Group similar lengths to reduce padding; restore the original metadata order.
    order = sorted(range(len(texts)), key=lambda position: len(texts[position]), reverse=True)
    offset = 0
    mixed_precision = use_fp16 and device.type == "cuda"
    while offset < len(order):
        positions = order[offset:offset + effective_batch_size]
        retry_oom = False
        try:
            batch_vectors = embed_once([texts[position] for position in positions], mixed_precision)
        except torch.cuda.OutOfMemoryError as error:
            if device.type != "cuda":
                raise
            if len(positions) == 1:
                raise RuntimeError(
                    f"CUDA OOM for input {positions[0]} even at batch size 1. "
                    "Use a GPU with more VRAM; no chunk was skipped."
                ) from error
            effective_batch_size = max(1, len(positions) // 2)
            print(f"CUDA OOM; retrying with batch_size={effective_batch_size}")
            retry_oom = True
        except FloatingPointError:
            if not mixed_precision:
                raise
            mixed_precision = False
            print("Invalid FP16 embeddings; retrying this batch with FP32")
            continue
        except RuntimeError as error:
            message = str(error).lower()
            if "device-side assert" in message or "illegal memory access" in message:
                raise RuntimeError(
                    "CUDA context failed. Restart the Kaggle/Colab runtime and run from "
                    "the first cell; set debug_cuda=True for a synchronous traceback."
                ) from error
            raise
        if retry_oom:
            # Retry only after the failed forward's exception frames have been released.
            gc.collect()
            torch.cuda.empty_cache()
            continue
        vectors[positions] = batch_vectors
        offset += len(positions)
    return vectors


index = faiss.read_index(str(index_path))
if not isinstance(index, faiss.IndexIVFPQ) or not index.is_trained or index.ntotal != 0:
    raise ValueError("Use a trained, empty IndexIVFPQ from FAISS_IVFPQ_Index.ipynb")
if index.metric_type != faiss.METRIC_INNER_PRODUCT:
    raise ValueError("Expected the inner-product index trained on L2-normalized embeddings")
index.nprobe = min(16, index.nlist)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Embedding device: {device}")
if device.type == "cpu":
    if require_gpu:
        raise RuntimeError("CUDA is unavailable. Enable a GPU runtime before continuing.")
    print("CUDA is unavailable; embedding runs on CPU. Enable GPU in Kaggle/Colab for large data.")
else:
    try:
        torch.cuda.synchronize(device)
    except RuntimeError as error:
        raise RuntimeError("CUDA is not usable. Restart the runtime and enable GPU.") from error
    print(f"GPU: {torch.cuda.get_device_name(device)}")

model = SentenceTransformer(
    embedding_model, trust_remote_code=True, device="cpu",
    model_kwargs={"code_revision": model_code_revision},
    config_kwargs={"code_revision": model_code_revision},
)
model.float().eval()
auto_model = model[0].auto_model
prepare_gte_model(auto_model)
if model.get_sentence_embedding_dimension() != index.d:
    raise ValueError("Model embedding dimension differs from the trained FAISS index")
if model.max_seq_length > auto_model.config.max_position_embeddings:
    raise ValueError("Tokenizer sequence limit exceeds the GTE position limit")
if getattr(model, "default_prompt_name", None):
    raise ValueError("Expected the GTE checkpoint without a default embedding prompt")
effective_batch_size = batch_size
smoke_texts = ["Embedding smoke test.", "Kiểm tra embedding tiếng Việt với câu dài hơn."]
embedding_device = device
device = torch.device("cpu")
try:
    cpu_probe = encode_texts(smoke_texts)
finally:
    device = embedding_device
model.to(device)
probe = encode_texts(smoke_texts)
print(f"CPU and {device} embedding smoke tests passed: {probe.shape}")

faiss_on_gpu = False
if use_faiss_gpu and device.type == "cuda" and hasattr(faiss, "StandardGpuResources"):
    try:
        gpu_resources = faiss.StandardGpuResources()
        gpu_resources.setTempMemory(256 * 1024**2)
        gpu_options = faiss.GpuClonerOptions()
        gpu_options.useFloat16 = True
        gpu_index = faiss.index_cpu_to_gpu(gpu_resources, 0, index, gpu_options)
    except RuntimeError as error:
        if "device-side assert" in str(error).lower() or "illegal memory access" in str(error).lower():
            raise RuntimeError("FAISS CUDA context failed; restart the runtime") from error
        print(f"FAISS GPU conversion failed; using CPU: {error}")
    else:
        index = gpu_index
        faiss_on_gpu = True
print(f"FAISS device: {'GPU 0' if faiss_on_gpu else 'CPU'}")
if use_faiss_gpu and not faiss_on_gpu:
    print("FAISS add uses CPU because a compatible GPU build/device is unavailable.")

## 3. Embedding, add vector và ghi metadata

Lọc tài liệu theo khoảng `doc_id`, `status="ok"`, rồi lấy từng child có `retrieval_eligible` và nội dung không rỗng. Metadata giữ `doc_id`, `chunk_order`, `chunk_text`, `chunk_id`, `parent_id`, `url`; `start_char`/`end_char` đổi thành `char_start`/`char_end` để giữ cấu trúc cũ. Kiểm tra các trường bắt buộc trước khi embedding và báo kèm doc/chunk khi dữ liệu sai.

ID vector liên tiếp từ 0, đồng bộ với key trong metadata. Các chunk cùng `doc_id` vẫn có ID riêng. Metadata ghi từng record và kiểm tra số lượng theo batch; không đọc toàn bộ metadata vào RAM lúc lưu.

```json
{"0": {"doc_id": 101003, "chunk_order": 0, "char_start": 0, "char_end": 844, "chunk_text": "Nội dung chunk...", "chunk_id": "101003:c:0:0:844:paragraph-v3", "parent_id": "101003:p:0:0:4111:paragraph-v3", "url": "https://example.com/article"}}
```

`batch_size` là số text forward cùng lúc (tự giảm nếu OOM); `add_batch_size` là số record gom một lượt. Bộ đọc JSON giới hạn mỗi document ở 64 MiB ký tự để không tích lũy cả file nếu JSON hỏng; RAM phụ thuộc một document, batch và FAISS index. Sửa tham số hàm nếu dữ liệu có một document lớn hơn giới hạn này.

In [ ]:
def iter_merged_documents(
    data_file: TextIO, read_size: int = 64 * 1024,
    max_document_chars: int = 64 * 1024**2,
) -> Iterator[dict]:
    """Stream a JSON array in bounded blocks, independent of line formatting."""
    if read_size < 1 or max_document_chars < read_size:
        raise ValueError("Use 0 < read_size <= max_document_chars")
    decoder = json.JSONDecoder()
    buffer = ""
    position = 0
    eof = False
    document_number = 0

    def read_more() -> None:
        nonlocal buffer, position, eof
        buffer = buffer[position:]
        position = 0
        block = data_file.read(read_size)
        if block:
            buffer += block
        else:
            eof = True

    def skip_whitespace() -> None:
        nonlocal position
        while True:
            while position < len(buffer) and buffer[position] in " \t\r\n":
                position += 1
            if position < len(buffer) or eof:
                return
            read_more()

    read_more()
    if buffer.startswith("\ufeff"):
        position += 1
    skip_whitespace()
    if position >= len(buffer) or buffer[position] != "[":
        raise ValueError("Expected merged.json to contain a JSON array")
    position += 1
    skip_whitespace()
    if position < len(buffer) and buffer[position] == "]":
        position += 1
    else:
        while True:
            skip_whitespace()
            if position >= len(buffer) or buffer[position] != "{":
                raise ValueError(f"Expected a document object at document {document_number + 1}")
            while True:
                try:
                    document, end_position = decoder.raw_decode(buffer, position)
                except json.JSONDecodeError as error:
                    if eof:
                        raise ValueError(f"Invalid or incomplete JSON at document {document_number + 1}") from error
                    if len(buffer) - position >= max_document_chars:
                        raise ValueError(f"Document {document_number + 1} exceeds max_document_chars or has invalid JSON") from error
                    read_more()
                else:
                    if end_position - position > max_document_chars:
                        raise ValueError(f"Document {document_number + 1} exceeds max_document_chars")
                    position = end_position
                    break
            document_number += 1
            yield document
            skip_whitespace()
            if position >= len(buffer):
                raise ValueError("Missing closing ']' in merged.json")
            separator = buffer[position]
            position += 1
            if separator == "]":
                break
            if separator != ",":
                raise ValueError(f"Missing document separator after document {document_number}")
            skip_whitespace()
            if position < len(buffer) and buffer[position] == "]":
                raise ValueError("Trailing comma in merged.json")
    skip_whitespace()
    if position < len(buffer):
        raise ValueError("Unexpected content after merged.json array")


def iter_chunk_records(
    data_file: TextIO, start_doc_id: int = 0, end_doc_id: int = -1,
) -> Iterator[dict]:
    """Flatten eligible children and preserve the existing metadata field names."""
    if start_doc_id < 0 or (end_doc_id != -1 and end_doc_id < start_doc_id):
        raise ValueError("Invalid doc_id range")
    for document in iter_merged_documents(data_file):
        doc_id = document.get("doc_id")
        if type(doc_id) is not int:
            raise ValueError("Each document must have an integer doc_id")
        if doc_id < start_doc_id or (end_doc_id != -1 and doc_id > end_doc_id):
            continue
        if document.get("status") != "ok":
            continue
        if not isinstance(document.get("children"), list):
            raise ValueError(f"Expected a children list in document {doc_id}")
        if not isinstance(document.get("url"), str):
            raise ValueError(f"Expected a URL string in document {doc_id}")
        for child_number, chunk in enumerate(document["children"]):
            context = f"document {doc_id}, child {child_number}"
            if not isinstance(chunk, dict):
                raise ValueError(f"Expected a child object in {context}")
            if not chunk.get("retrieval_eligible", True):
                continue
            text = chunk.get("chunk_text")
            if not isinstance(text, str):
                raise ValueError(f"Non-string chunk_text in {context}")
            if not text.strip():
                continue
            required = ("doc_id", "chunk_order", "start_char", "end_char", "chunk_id", "parent_id")
            missing = [name for name in required if name not in chunk]
            if missing:
                raise ValueError(f"Missing fields {missing} in {context}")
            if type(chunk["doc_id"]) is not int or chunk["doc_id"] != doc_id:
                raise ValueError(f"Child doc_id differs from document in {context}")
            for name in ("chunk_order", "start_char", "end_char"):
                if type(chunk[name]) is not int or chunk[name] < 0:
                    raise ValueError(f"Expected a non-negative integer {name} in {context}")
            if chunk["end_char"] < chunk["start_char"]:
                raise ValueError(f"end_char precedes start_char in {context}")
            for name in ("chunk_id", "parent_id"):
                if not isinstance(chunk[name], str) or not chunk[name]:
                    raise ValueError(f"Expected a non-empty string {name} in {context}")
            yield {
                "doc_id": doc_id,
                "chunk_order": chunk["chunk_order"],
                "char_start": chunk["start_char"],
                "char_end": chunk["end_char"],
                "chunk_text": text,
                "chunk_id": chunk["chunk_id"],
                "parent_id": chunk["parent_id"],
                "url": document["url"],
            }


def add_batch(records: list[dict], metadata_file: TextIO, metadata_count: int) -> int:
    """Add sequential vector IDs and stream matching metadata without whole-file reads."""
    if not records:
        return metadata_count
    if index.ntotal != metadata_count:
        raise ValueError("FAISS vector count differs from metadata count before adding")
    vectors = encode_texts([record["chunk_text"] for record in records])
    ids = np.arange(metadata_count, metadata_count + len(records), dtype=np.int64)
    index.add_with_ids(vectors, ids)
    if index.ntotal != metadata_count + len(records):
        raise ValueError("FAISS did not add the expected number of vectors")
    for vector_id, record in zip(ids, records):
        if metadata_count:
            metadata_file.write(",\n")
        metadata_file.write(
            f'"{int(vector_id)}":' + json.dumps(record, ensure_ascii=False, separators=(",", ":"))
        )
        metadata_count += 1
    metadata_file.flush()
    print(f"\rVectors added: {index.ntotal:,}", end="", flush=True)
    return metadata_count


def build_chunk_index() -> int:
    """Build once per loaded empty index; keep incomplete metadata explicitly marked."""
    if index.ntotal != 0:
        raise ValueError("Reload an empty trained index before rerunning the build cell")
    for path in (output_index, output_metadata, partial_index, partial_metadata):
        if path.exists():
            raise FileExistsError(f"Choose a new output_dir before rerunning: {path}")
    count = 0
    batch = []
    with data_path.open(encoding="utf-8-sig") as data_file, partial_metadata.open("x", encoding="utf-8") as metadata_file:
        metadata_file.write("{\n")
        for record in iter_chunk_records(data_file, start_point, end_point):
            batch.append(record)
            if len(batch) >= add_batch_size:
                count = add_batch(batch, metadata_file, count)
                batch.clear()
        if batch:
            count = add_batch(batch, metadata_file, count)
        metadata_file.write("\n}\n")
    if count != index.ntotal:
        raise ValueError("FAISS vector count differs from metadata count")
    print(f"\nTotal vectors: {count:,}")
    return count


build_complete = False
metadata_count = build_chunk_index()
build_complete = True

## 4. Lưu index sau khi add

Chỉ lưu khi cell build đã hoàn tất và số metadata khớp `index.ntotal`. Chuyển index GPU về CPU nếu cần. Ghi `.faiss.partial`, đọc lại header/số lượng để kiểm tra rồi công bố hai output bằng tên cuối, không ghi đè file sẵn có. Không dùng `json.load()` trên metadata lớn.

Sau khi thành công, `metadata[str(i)]` là record tương ứng ID search `i`. Với metadata rất lớn, ứng dụng cần cách tra cứu phù hợp thay vì tải toàn bộ JSON vào RAM; cấu trúc output vẫn giữ tương thích notebook cũ.

In [ ]:
def save_chunk_index() -> None:
    """Publish a completed index/metadata pair without overwriting existing files."""
    if not build_complete or metadata_count != index.ntotal:
        raise ValueError("Run the build cell successfully before saving")
    if not partial_metadata.is_file():
        raise FileNotFoundError("Completed partial metadata is missing")
    for path in (output_index, output_metadata, partial_index):
        if path.exists():
            raise FileExistsError(f"Output already exists: {path}")
    index_to_save = faiss.index_gpu_to_cpu(index) if faiss_on_gpu else index
    # Exclusive reservation prevents accidental overwriting of a previous partial index.
    with partial_index.open("xb"):
        pass
    faiss.write_index(index_to_save, str(partial_index))
    # mmap validates the saved index without copying all inverted lists into RAM.
    saved_index = faiss.read_index(str(partial_index), faiss.IO_FLAG_MMAP)
    if saved_index.ntotal != metadata_count or saved_index.d != index.d or not saved_index.is_trained:
        raise ValueError("Saved FAISS index failed dimension/count/training validation")
    del saved_index
    # Same-directory rename also works on Drive mounts that do not support hard links.
    for partial_path, final_path in ((partial_index, output_index), (partial_metadata, output_metadata)):
        if final_path.exists():
            raise FileExistsError(f"Output already exists: {final_path}")
        partial_path.rename(final_path)
    print(f"Index: {output_index}")
    print(f"Metadata: {output_metadata}")


save_chunk_index()